In [3]:
import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Embedding
)

from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. UCI-HAR Download
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"

if os.path.exists(DATA_PATH):

    print("Existing dataset found")
    print(DATA_PATH)

else:

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(
            "/content/drive/MyDrive/Colab Notebooks"
        )

    print("Download completed")


# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # --------------------------------------------------------
    # (9, N, 128)
    #        ↓
    # (N, 128, 9)
    # --------------------------------------------------------

    X = np.stack(X, axis=-1)

    y_path = (
        f"{DATA_PATH}/{split}/"
        f"y_{split}.txt"
    )

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    return X, y


# Official train/test split
X_train, y_train = load_data("train")
X_test, y_test = load_data("test")


print("\nDataset shape")

print(
    "X_train:",
    X_train.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "X_test :",
    X_test.shape
)

print(
    "y_test :",
    y_test.shape
)


# ============================================================
# 5. Normalization
# ============================================================

# IMPORTANT:
# CNN과 동일한 정규화 방식

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)


X_train = (
    X_train - mean
) / (
    std + 1e-8
)


X_test = (
    X_test - mean
) / (
    std + 1e-8
)


# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6


y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)


y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)


# ============================================================
# 7. Transformer Encoder Block
# ============================================================

def transformer_encoder(
    inputs,
    head_size=64,
    num_heads=4,
    ff_dim=128,
    dropout=0.1
):

    # --------------------------------------------------------
    # Layer Normalization
    # --------------------------------------------------------

    x = LayerNormalization(
        epsilon=1e-6
    )(inputs)


    # --------------------------------------------------------
    # Multi-Head Self Attention
    # --------------------------------------------------------

    x = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(x, x)


    x = Dropout(
        dropout
    )(x)


    # --------------------------------------------------------
    # Residual Connection
    # --------------------------------------------------------

    x = x + inputs


    # --------------------------------------------------------
    # Feed Forward Network
    # --------------------------------------------------------

    y = LayerNormalization(
        epsilon=1e-6
    )(x)


    y = Dense(
        ff_dim,
        activation="gelu"
    )(y)


    y = Dropout(
        dropout
    )(y)


    y = Dense(
        inputs.shape[-1]
    )(y)


    y = Dropout(
        dropout
    )(y)


    # --------------------------------------------------------
    # Residual Connection
    # --------------------------------------------------------

    return x + y


# ============================================================
# 8. Transformer Model
# ============================================================

inputs = Input(
    shape=(128, 9)
)


# ============================================================
# Input Projection
# ============================================================

x = Dense(
    128
)(inputs)


# ============================================================
# Positional Embedding
# ============================================================

positions = tf.range(
    start=0,
    limit=128,
    delta=1
)


position_embedding = Embedding(
    input_dim=128,
    output_dim=128
)


position_embeddings = position_embedding(
    positions
)


x = x + position_embeddings


# ============================================================
# Transformer Encoder 1
# ============================================================

x = transformer_encoder(
    x,
    head_size=64,
    num_heads=4,
    ff_dim=256,
    dropout=0.1
)


# ============================================================
# Transformer Encoder 2
# ============================================================

x = transformer_encoder(
    x,
    head_size=64,
    num_heads=4,
    ff_dim=256,
    dropout=0.1
)


# ============================================================
# Global Average Pooling
# ============================================================

x = GlobalAveragePooling1D()(x)


# ============================================================
# Dense Layer
# ============================================================

x = Dense(
    128,
    activation="gelu"
)(x)


# ============================================================
# Dropout
# ============================================================

x = Dropout(
    0.3
)(x)


# ============================================================
# Output
# ============================================================

outputs = Dense(
    n_outputs,
    activation="softmax"
)(x)


# ============================================================
# 9. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


# ============================================================
# 10. Compile
# ============================================================

model.compile(
    loss="categorical_crossentropy",

    optimizer="adam",

    metrics=[
        "accuracy"
    ]
)


# ============================================================
# 11. Model Summary
# ============================================================

model.summary()


# ============================================================
# 12. Train
# ============================================================

history = model.fit(

    X_train,

    y_train_onehot,

    # CNN과 동일
    epochs=20,

    # CNN과 동일
    batch_size=64,

    # CNN과 동일
    validation_split=0.2,

    # CNN과 동일
    shuffle=True,

    # CNN과 동일
    verbose=1
)


# ============================================================
# 13. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)


y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 14. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)


precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


# ============================================================
# 15. Test Results
# ============================================================

print("\n==============================")
print("TEST RESULTS")
print("==============================")


print(
    f"Accuracy  : {accuracy:.4f}"
)


print(
    f"Precision : {precision:.4f}"
)


print(
    f"Recall    : {recall:.4f}"
)


print(
    f"F1-score  : {f1:.4f}"
)


# ============================================================
# 16. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")


print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 17. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)


print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")


print(cm)

Existing dataset found
/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset

Dataset shape
X_train: (7352, 128, 9)
y_train: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 128, 128)  │      1,280 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_14 (Add)        │ (None, 128, 128)  │          0 │ dense_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 128, 128)  │        256 │ add_14[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 128, 128)  │    131,968 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_27          │ (None, 128, 128)  │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_15 (Add)        │ (None, 128, 128)  │          0 │ dropout_27[0][0], │
│                     │                   │            │ add_14[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 128, 128)  │        256 │ add_15[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 128, 256)  │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_28          │ (None, 128, 256)  │          0 │ dense_19[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 128, 128)  │     32,896 │ dropout_28[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_29          │ (None, 128, 128)  │          0 │ dense_20[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_16 (Add)        │ (None, 128, 128)  │          0 │ add_15[0][0],     │
│                     │                   │            │ dropout_29[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 128, 128)  │        256 │ add_16[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 128, 128)  │    131,968 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_31          │ (None, 128, 128)  │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_17 (Add)        │ (None, 128, 128)  │          0 │ dropout_31[0][0], │
│                     │                   │            │ add_16[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 128, 128)  │        256 │ add_17[0][0]      │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 415,366 (1.58 MB)

 Trainable params: 415,366 (1.58 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 39s 163ms/step - accuracy: 0.8597 - loss: 0.3617 - val_accuracy: 0.8742 - val_loss: 0.3943
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9403 - loss: 0.1526 - val_accuracy: 0.8838 - val_loss: 0.4229
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9539 - loss: 0.1134 - val_accuracy: 0.8729 - val_loss: 0.5477
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9543 - loss: 0.1067 - val_accuracy: 0.9062 - val_loss: 0.3998
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9580 - loss: 0.0917 - val_accuracy: 0.9171 - val_loss: 0.3789
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9606 - loss: 0.0898 - val_accuracy: 0.9116 - val_loss: 0.4263
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9541 - loss: 0.1087 - val_accuracy: 0.8926 - val_loss: 0.4474
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9531 - loss: 0.1063 - val_accuracy: 0.8817 -

In [3]:
import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
    GlobalAveragePooling1D,
    Add
)

from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. UCI-HAR Download
# ============================================================

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset"

if os.path.exists(DATA_PATH):

    print("Existing dataset found")
    print(DATA_PATH)

else:

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/machine-learning-databases/"
        "00240/UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/drive/MyDrive/Colab Notebooks")

    print("Download completed")


# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    return X, y


# Official train/test split
X_train, y_train = load_data("train")
X_test, y_test = load_data("test")


print("\nDataset shape")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


# ============================================================
# 5. Normalization
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (
    X_train - mean
) / (
    std + 1e-8
)

X_test = (
    X_test - mean
) / (
    std + 1e-8
)


# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)


# ============================================================
# 7. Transformer Encoder Block
# ============================================================

def transformer_encoder(
    inputs,
    head_size=64,
    num_heads=4,
    ff_dim=128,
    dropout=0.1
):

    # --------------------------------------------------------
    # Multi-Head Self Attention
    # --------------------------------------------------------

    x = LayerNormalization(
        epsilon=1e-6
    )(inputs)

    attention_output = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads,
        dropout=dropout
    )(
        x,
        x
    )

    attention_output = Dropout(
        dropout
    )(attention_output)

    # Residual connection
    x = Add()(
        [inputs, attention_output]
    )


    # --------------------------------------------------------
    # Feed Forward Network
    # --------------------------------------------------------

    y = LayerNormalization(
        epsilon=1e-6
    )(x)

    y = Dense(
        ff_dim,
        activation="gelu"
    )(y)

    y = Dropout(
        dropout
    )(y)

    y = Dense(
        inputs.shape[-1]
    )(y)

    y = Dropout(
        dropout
    )(y)

    # Residual connection
    outputs = Add()(
        [x, y]
    )

    return outputs


# ============================================================
# 8. CNN + Transformer Model
# ============================================================

inputs = Input(
    shape=(128, 9)
)


# ============================================================
# CNN Feature Extraction
# ============================================================

conv1 = Conv1D(
    filters=64,
    kernel_size=5,
    padding="same",
    activation="relu"
)(inputs)


pool1 = MaxPooling1D(
    pool_size=2
)(conv1)


conv2 = Conv1D(
    filters=128,
    kernel_size=5,
    padding="same",
    activation="relu"
)(pool1)


pool2 = MaxPooling1D(
    pool_size=2
)(conv2)


# ============================================================
# CNN Output
#
# (128, 9)
#      ↓
# (64, 64)
#      ↓
# (32, 128)
#
# Transformer input:
# (batch, 32, 128)
# ============================================================


# ============================================================
# Transformer Encoder
# ============================================================

transformer = transformer_encoder(
    pool2,
    head_size=64,
    num_heads=4,
    ff_dim=256,
    dropout=0.1
)


# Second Transformer Encoder
transformer = transformer_encoder(
    transformer,
    head_size=64,
    num_heads=4,
    ff_dim=256,
    dropout=0.1
)


# ============================================================
# Global Average Pooling
# ============================================================

gap = GlobalAveragePooling1D()(
    transformer
)


# ============================================================
# Classification Head
# ============================================================

dense = Dense(
    128,
    activation="relu"
)(gap)


dense = Dropout(
    0.3
)(dense)


outputs = Dense(
    n_outputs,
    activation="softmax"
)(dense)


# ============================================================
# 9. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


model.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    metrics=["accuracy"]
)


model.summary()


# ============================================================
# 10. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,

    epochs=30,

    batch_size=64,

    validation_split=0.2,

    shuffle=True,

    verbose=1
)


# ============================================================
# 11. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 12. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("TEST RESULTS")
print("==============================")

print(
    f"Accuracy  : {accuracy:.4f}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1-score  : {f1:.4f}"
)


# ============================================================
# 13. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]


print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================")


print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 14. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)


print("\n==============================")
print("CONFUSION MATRIX")
print("==============================")


print(cm)

Existing dataset found
/content/drive/MyDrive/Colab Notebooks/UCI HAR Dataset

Dataset shape
X_train: (7352, 128, 9)
y_train: (7352,)
X_test : (2947, 128, 9)
y_test : (2947,)


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 128, 64)   │      2,944 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_4     │ (None, 64, 64)    │          0 │ conv1d_4[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 64, 128)   │     41,088 │ max_pooling1d_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_5     │ (None, 32, 128)   │          0 │ conv1d_5[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 128)   │        256 │ max_pooling1d_5[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 32, 128)   │    131,968 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 32, 128)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 32, 128)   │          0 │ max_pooling1d_5[… │
│                     │                   │            │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 128)   │        256 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 32, 256)   │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 32, 256)   │          0 │ dense_10[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32, 128)   │     32,896 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 32, 128)   │          0 │ dense_11[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 32, 128)   │          0 │ add_6[0][0],      │
│                     │                   │            │ dropout_16[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 128)   │        256 │ add_7[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 32, 128)   │    131,968 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 32, 128)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                 

 Total params: 458,118 (1.75 MB)

 Trainable params: 458,118 (1.75 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 28s 143ms/step - accuracy: 0.8529 - loss: 0.3839 - val_accuracy: 0.9191 - val_loss: 0.2380
Epoch 2/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9539 - loss: 0.1050 - val_accuracy: 0.9205 - val_loss: 0.2455
Epoch 3/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9505 - loss: 0.1137 - val_accuracy: 0.9205 - val_loss: 0.2729
Epoch 4/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9617 - loss: 0.0792 - val_accuracy: 0.9307 - val_loss: 0.4231
Epoch 5/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9604 - loss: 0.0823 - val_accuracy: 0.9341 - val_loss: 0.3700
Epoch 6/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9646 - loss: 0.0704 - val_accuracy: 0.9279 - val_loss: 0.4526
Epoch 7/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9633 - loss: 0.0703 - val_accuracy: 0.9171 - val_loss: 0.4213
Epoch 8/30
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9740 - loss: 0.0568 - val_accuracy: 0.9395 - val_l